# 06 - Spatio-Temporal Crime Prediction (GNN + Symbolic Rules)

This notebook builds a **neuro-symbolic crime prediction** pipeline that combines:

- **Graph Neural Networks (GNN)** for learning spatial diffusion patterns across
  police precincts (*seccionales*)
- **Symbolic temporal rules** encoding domain knowledge about seasonality,
  escalation thresholds, and facility proximity

The approach follows the **LLM + Symbolic** integration pattern: neural
components learn spatial representations from data while symbolic rules inject
domain expertise that would be difficult to learn from limited training samples.

In [ ]:
!pip install -q torch torch-geometric polars geopandas networkx matplotlib pyarrow

In [ ]:
from pathlib import Path
from typing import Optional

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GATConv

In [ ]:
# ---------------------------------------------------------------------------
# Load processed seccionales geoparquet and crime data
# ---------------------------------------------------------------------------
DATA_DIR = Path("../data/processed")

GEO_PATH = DATA_DIR / "seccionales.parquet"
CRIME_PATH = DATA_DIR / "delitos_denunciados.parquet"

# Load or create synthetic geodata
if GEO_PATH.exists():
    gdf = gpd.read_parquet(GEO_PATH)
    print(f"Seccionales loaded: {len(gdf)} precincts")
else:
    print(f"[INFO] {GEO_PATH} not found - creating synthetic grid.")
    from shapely.geometry import box

    # Create a 4x4 grid of synthetic precincts
    polys = []
    ids = []
    for i in range(4):
        for j in range(4):
            polys.append(box(i, j, i + 1, j + 1))
            ids.append(f"SEC_{i*4+j+1:02d}")
    gdf = gpd.GeoDataFrame(
        {"seccional_id": ids, "geometry": polys},
        crs="EPSG:4326",
    )
    print(f"Synthetic grid: {len(gdf)} precincts")

# Load or create synthetic crime data
if CRIME_PATH.exists():
    df_crime = pl.read_parquet(CRIME_PATH)
    print(f"Crime data loaded: {df_crime.shape}")
else:
    print(f"[INFO] {CRIME_PATH} not found - creating synthetic crime data.")
    rng = np.random.default_rng(42)
    n_precincts = len(gdf)
    n_months = 24  # 2 years of monthly data
    crime_types = ["Hurto", "Rapiña", "Homicidio", "Lesiones", "Violencia domestica"]

    records = []
    for sec_idx in range(n_precincts):
        for month_idx in range(n_months):
            year = 2022 + month_idx // 12
            month = (month_idx % 12) + 1
            for crime in crime_types:
                base_rate = rng.poisson(50)
                # Add seasonality: summer (Dec-Feb) bump
                if month in (12, 1, 2):
                    base_rate = int(base_rate * 1.3)
                records.append({
                    "seccional_id": gdf["seccional_id"].iloc[sec_idx],
                    "anio": year,
                    "mes": month,
                    "delito": crime,
                    "cantidad": base_rate,
                })

    df_crime = pl.DataFrame(records)
    print(f"Synthetic crime data: {df_crime.shape}")

gdf.head()

In [ ]:
# ---------------------------------------------------------------------------
# Build seccional adjacency graph from polygon boundaries
# ---------------------------------------------------------------------------
# Two precincts are adjacent if their polygons share a boundary (touch/overlap).

G = nx.Graph()
sec_ids = gdf["seccional_id"].tolist()
id_to_idx = {sid: i for i, sid in enumerate(sec_ids)}

for sid in sec_ids:
    G.add_node(sid)

# Find adjacent polygons
for i in range(len(gdf)):
    for j in range(i + 1, len(gdf)):
        if gdf.geometry.iloc[i].touches(gdf.geometry.iloc[j]) or \
           gdf.geometry.iloc[i].intersects(gdf.geometry.iloc[j]):
            # Exclude containment (only shared edges)
            shared = gdf.geometry.iloc[i].intersection(gdf.geometry.iloc[j])
            if shared.length > 0:  # shared edge, not just a point
                G.add_edge(sec_ids[i], sec_ids[j])

print(f"Adjacency graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Average degree: {2 * G.number_of_edges() / G.number_of_nodes():.1f}")

# Convert to PyG edge_index
edge_list = []
for u, v in G.edges():
    ui, vi = id_to_idx[u], id_to_idx[v]
    edge_list.append([ui, vi])
    edge_list.append([vi, ui])  # undirected

edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
print(f"edge_index shape: {edge_index.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Create node features: crime counts per type per seccional
# ---------------------------------------------------------------------------
# Feature vector per precinct: [count_type1, count_type2, ..., count_typeN]
# aggregated over the most recent available period.

# Detect column names
col_sec = next((c for c in df_crime.columns if "seccional" in c.lower()), None)
col_crime_type = next(
    (c for c in df_crime.columns if c.lower() in ("delito", "tipo", "crime_type")),
    "delito",
)
col_count = next(
    (c for c in df_crime.columns if c.lower() in ("cantidad", "count", "total")),
    "cantidad",
)

if col_sec is None:
    # If no seccional column, use the first available identifier
    col_sec = "seccional_id"

crime_types = sorted(df_crime[col_crime_type].unique().to_list())
print(f"Crime types: {crime_types}")

# Pivot: rows=seccionales, columns=crime types, values=total counts
pivot = (
    df_crime.group_by([col_sec, col_crime_type])
    .agg(pl.col(col_count).sum())
    .pivot(on=col_crime_type, index=col_sec, values=col_count)
    .fill_null(0)
    .sort(col_sec)
)

# Build feature matrix aligned with node ordering
feature_cols = [c for c in pivot.columns if c != col_sec]
n_nodes = len(sec_ids)
n_features = len(feature_cols)

X = np.zeros((n_nodes, n_features), dtype=np.float32)
pivot_dict = {row[col_sec]: row for row in pivot.iter_rows(named=True)}

for sid, idx in id_to_idx.items():
    if sid in pivot_dict:
        row = pivot_dict[sid]
        X[idx] = [row.get(c, 0) for c in feature_cols]

# Normalize features
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True) + 1e-8
X_norm = (X - X_mean) / X_std

x_tensor = torch.tensor(X_norm, dtype=torch.float)
print(f"Node feature matrix: {x_tensor.shape} (nodes x crime_types)")

In [ ]:
# ---------------------------------------------------------------------------
# Symbolic Temporal Rules
# ---------------------------------------------------------------------------
# Domain knowledge encoded as deterministic rules that augment neural predictions.


def seasonal_adjustment(month: int) -> float:
    """Seasonal crime pattern for Uruguay.

    Summer months (December-February) show elevated crime rates.
    Winter months (June-August) show reduced rates.

    Returns a multiplicative factor.
    """
    SEASONAL_FACTORS = {
        1: 1.25,   # January - summer, high
        2: 1.20,   # February - summer
        3: 1.10,   # March - autumn transition
        4: 1.00,   # April - baseline
        5: 0.95,   # May - cooling
        6: 0.90,   # June - winter, low
        7: 0.88,   # July - winter, lowest
        8: 0.90,   # August - winter
        9: 0.95,   # September - spring transition
        10: 1.00,  # October - baseline
        11: 1.10,  # November - warming
        12: 1.30,  # December - summer, highest
    }
    return SEASONAL_FACTORS.get(month, 1.0)


def escalation_rule(
    dv_rate: float,
    dv_threshold: float = 100.0,
    has_cevdg_nearby: bool = False,
) -> float:
    """Escalation rule: elevated risk when DV rate exceeds threshold
    and no CEVDG (Centro Especializado en Violencia Domestica y de Genero)
    is nearby.

    Returns a risk multiplier.
    """
    if dv_rate > dv_threshold and not has_cevdg_nearby:
        return 1.5  # elevated risk
    elif dv_rate > dv_threshold and has_cevdg_nearby:
        return 1.1  # moderate risk (facility mitigates)
    return 1.0      # baseline


# Demo: apply seasonal adjustment for each month
print("Seasonal adjustment factors (Uruguay):")
for m in range(1, 13):
    factor = seasonal_adjustment(m)
    bar = "#" * int(factor * 20)
    print(f"  Month {m:2d}: {factor:.2f} {bar}")

print("\nEscalation rule examples:")
print(f"  DV=150, no CEVDG: {escalation_rule(150, has_cevdg_nearby=False):.1f}x")
print(f"  DV=150, CEVDG:    {escalation_rule(150, has_cevdg_nearby=True):.1f}x")
print(f"  DV=50,  no CEVDG: {escalation_rule(50, has_cevdg_nearby=False):.1f}x")

In [ ]:
# ---------------------------------------------------------------------------
# GNN Model Definition
# ---------------------------------------------------------------------------
# We use a Graph Convolutional Network (GCN) with optional GAT attention
# to learn spatial diffusion of crime patterns across the precinct graph.


class CrimeGCN(torch.nn.Module):
    """Graph Convolutional Network for crime prediction.

    Input:  node features (crime counts per type per precinct)
    Output: predicted total crime count per precinct (regression)
    """

    def __init__(self, in_channels: int, hidden_channels: int = 32):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.linear = torch.nn.Linear(hidden_channels, 1)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=0.3, training=self.training)
        h = self.conv2(h, edge_index)
        h = F.relu(h)
        out = self.linear(h).squeeze(-1)
        return out


class CrimeGAT(torch.nn.Module):
    """Graph Attention Network variant for crime prediction.

    Uses attention to weight neighbor contributions differently,
    which may capture asymmetric crime diffusion patterns.
    """

    def __init__(self, in_channels: int, hidden_channels: int = 32, heads: int = 4):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1)
        self.linear = torch.nn.Linear(hidden_channels, 1)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = self.conv1(x, edge_index)
        h = F.elu(h)
        h = F.dropout(h, p=0.3, training=self.training)
        h = self.conv2(h, edge_index)
        h = F.elu(h)
        out = self.linear(h).squeeze(-1)
        return out


# Instantiate model
model = CrimeGCN(in_channels=n_features, hidden_channels=32)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ---------------------------------------------------------------------------
# Training Loop (train/val split by time)
# ---------------------------------------------------------------------------
# Target: total crime count per precinct (sum across types).
# In a real scenario we would split by time (train on 2022, validate on 2023).
# Here we use synthetic targets for demonstration.

# Create target: total crime per precinct
y_raw = X.sum(axis=1)  # sum across crime types
y_mean = y_raw.mean()
y_std = y_raw.std() + 1e-8
y_norm = (y_raw - y_mean) / y_std
y_tensor = torch.tensor(y_norm, dtype=torch.float)

# Train/val split (first 75% train, last 25% val - simulating temporal split)
n_train = int(0.75 * n_nodes)
train_mask = torch.zeros(n_nodes, dtype=torch.bool)
train_mask[:n_train] = True
val_mask = ~train_mask

# Build PyG data object
data = Data(x=x_tensor, edge_index=edge_index, y=y_tensor)
data.train_mask = train_mask
data.val_mask = val_mask

# Training
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
loss_fn = torch.nn.MSELoss()

train_losses = []
val_losses = []

model.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)

    train_loss = loss_fn(out[data.train_mask], data.y[data.train_mask])
    train_loss.backward()
    optimizer.step()

    with torch.no_grad():
        val_loss = loss_fn(out[data.val_mask], data.y[data.val_mask])

    train_losses.append(train_loss.item())
    val_losses.append(val_loss.item())

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch+1:3d} | "
            f"Train Loss: {train_loss.item():.4f} | "
            f"Val Loss: {val_loss.item():.4f}"
        )

# Plot training curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label="Train Loss")
ax.plot(val_losses, label="Val Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("GCN Training Curve")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Neuro-Symbolic Fusion: Combine Neural + Symbolic Predictions
# ---------------------------------------------------------------------------
# The GNN provides a spatial crime prediction.
# Symbolic rules provide temporal adjustments and escalation modifiers.
# The final prediction combines both.

model.eval()
with torch.no_grad():
    neural_pred_norm = model(data.x, data.edge_index).numpy()

# Denormalize
neural_pred = neural_pred_norm * y_std + y_mean

# Apply symbolic rules for a specific prediction scenario
PREDICTION_MONTH = 1   # January (summer in Uruguay)
seasonal_factor = seasonal_adjustment(PREDICTION_MONTH)

# Simulate DV rates and facility presence per precinct
rng = np.random.default_rng(123)
dv_rates = rng.poisson(80, size=n_nodes).astype(float)
has_cevdg = rng.choice([True, False], size=n_nodes, p=[0.3, 0.7])

escalation_factors = np.array(
    [escalation_rule(dv, has_cevdg_nearby=cevdg) for dv, cevdg in zip(dv_rates, has_cevdg)]
)

# Neuro-symbolic fusion
combined_pred = neural_pred * seasonal_factor * escalation_factors

# Display results
print(f"Prediction month: {PREDICTION_MONTH} (seasonal factor: {seasonal_factor:.2f})")
print(f"{'Precinct':>12s} | {'Neural':>8s} | {'Seasonal':>8s} | {'Escalation':>10s} | {'Combined':>10s}")
print("-" * 65)
for i in range(min(10, n_nodes)):
    print(
        f"{sec_ids[i]:>12s} | "
        f"{neural_pred[i]:8.1f} | "
        f"{seasonal_factor:8.2f} | "
        f"{escalation_factors[i]:10.2f} | "
        f"{combined_pred[i]:10.1f}"
    )

In [ ]:
# ---------------------------------------------------------------------------
# Visualize Predictions on Map
# ---------------------------------------------------------------------------

gdf_pred = gdf.copy()
gdf_pred["neural_pred"] = neural_pred
gdf_pred["combined_pred"] = combined_pred
gdf_pred["escalation"] = escalation_factors

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Neural prediction
gdf_pred.plot(
    column="neural_pred",
    cmap="YlOrRd",
    legend=True,
    ax=axes[0],
    edgecolor="black",
    linewidth=0.5,
)
axes[0].set_title("GNN Spatial Prediction")
axes[0].set_axis_off()

# Escalation factor
gdf_pred.plot(
    column="escalation",
    cmap="RdYlGn_r",
    legend=True,
    ax=axes[1],
    edgecolor="black",
    linewidth=0.5,
)
axes[1].set_title("Symbolic Escalation Factor")
axes[1].set_axis_off()

# Combined neuro-symbolic
gdf_pred.plot(
    column="combined_pred",
    cmap="YlOrRd",
    legend=True,
    ax=axes[2],
    edgecolor="black",
    linewidth=0.5,
)
axes[2].set_title(f"Neuro-Symbolic (Month {PREDICTION_MONTH})")
axes[2].set_axis_off()

plt.suptitle("Crime Prediction: Neural vs Symbolic vs Combined", fontsize=14)
plt.tight_layout()
plt.show()

## Summary: Neuro-Symbolic Crime Prediction

This notebook demonstrated the **LLM + Symbolic** integration pattern applied
to spatio-temporal crime prediction across Uruguayan police precincts.

### Architecture

| Component | Type | Role |
|---|---|---|
| **GCN/GAT** | Neural | Learn spatial diffusion patterns from precinct adjacency graph |
| **Seasonal rules** | Symbolic | Inject known temporal patterns (Dec-Feb crime increase in Uruguay) |
| **Escalation rules** | Symbolic | Encode domain knowledge (DV threshold + facility proximity) |
| **Fusion** | Hybrid | Multiply neural predictions by symbolic adjustment factors |

### Why Neuro-Symbolic?

1. **Data efficiency**: Symbolic rules encode domain knowledge that would require
   years of data for a purely neural model to learn (seasonality, facility effects).

2. **Interpretability**: Each prediction can be decomposed into neural (spatial)
   and symbolic (temporal, escalation) components, making the model auditable
   for law enforcement stakeholders.

3. **Robustness**: Symbolic rules provide hard constraints that prevent the neural
   component from making predictions that violate domain invariants.

### Limitations and Next Steps

- **Data**: Production deployment needs real precinct boundaries and multi-year
  crime records; this notebook uses synthetic data for demonstration.
- **Temporal GNN**: Extend to temporal graph networks (T-GCN, DCRNN) for
  learning time-varying spatial patterns directly.
- **LLM integration**: Use an LLM to *discover* new symbolic rules from
  unstructured incident reports (completing the NeSy loop).
- **Evaluation**: Apply proper backtesting with temporal cross-validation.